In [1]:
#   numpy(넘파이)는 숫자 여러 개를 '배열(array)'로 묶어 한 번에 계산하게 해 주는 라이브러리입니다.
#   예) [160, 170, 180]에 모두 2를 더하는 계산을 한 줄로 할 수 있습니다.
#   이 노트북에서는 데이터 준비/정규화 같은 'NumPy 흐름'에 사용합니다.
import numpy as np

#   torch는 PyTorch 라이브러리입니다.
#   이번 노트북에서는 다음을 쓰기 위해 사용합니다.
#       (1) torch.nn.Module : PyTorch 모델을 만들기 위한 기본 class
#       (2) torch.nn.Linear : H(x)=a·X_norm + b 계산을 대신해 주는 부품(model 안에 넣음)
#       (3) torch.nn.BCELoss: Binary Cross Entropy Cost 계산을 대신해 주는 부품
#       (4) 자동 미분(autograd)과 optimizer(torch.optim.SGD)
#
#   참고: 이 노트북에서는 'import torch.nn as nn' 같은 줄임 import를 쓰지 않습니다.
#       항상 torch.nn.Linear, torch.nn.BCELoss 처럼 전체 이름을 그대로 적습니다.
#       그래야 이 기능들이 torch.nn 안에 있는 API라는 사실이 코드에 그대로 보입니다.
import torch

In [2]:
#   ========================================
#   1. 입력값 X와 정답 y 준비   (이전 파일과 동일)
#   ========================================

#   X: 입력값입니다. 여기서는 사람의 키(cm)를 사용합니다.
#       np.array([...]) 는 여러 숫자를 하나의 NumPy 배열로 묶는 것입니다.
X = np.array([160, 170, 180, 190])

#   y: 정답값입니다. 0은 농구선수 아님, 1은 농구선수입니다.
#       X와 y는 순서대로 짝지어집니다. 즉 키 160 → 정답 0, 키 190 → 정답 1.
y = np.array([0, 0, 1, 1])

print('입력값 X: ', X)
print('입력값 y: ', y)

입력값 X:  [160 170 180 190]
입력값 y:  [0 0 1 1]


In [3]:
#   ========================================
#   2. 입력값 정규화    (이전 파일과 동일)
#   ========================================

#   평균(mean)과 표준편차(std)를 계산합니다.
#   - 평균      : 데이터의 중심(가운데쯤 되는 값)
#   - 표준편차   : 데이터가 평균에서 얼마나 넓게 퍼져 있는지를 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

#   정규화 공식: (원본값 - 평균) / 표준편차
#   주의: 실제 학습에는 원래 키 X가 아니라, 정규화된 입력값 X_norm을 사용합니다.
#         (X_mean, X_std는 나중에 '새 입력값 예측'에서도 똑같이 다시 씁니다.)
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean: ', X_mean)
print('입력값 표준편차 X_std: ', X_std)
print('정규화된 입력값 X_norm: ', X_norm)

입력값 평균 X_mean:  175.0
입력값 표준편차 X_std:  11.180339887498949
정규화된 입력값 X_norm:  [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
#   ========================================
#   2-1. X_norm 과 y를 PyTorch tensor 로 변환하고 shape을 (n, 1)로 정리
#   ========================================

#   dtype = torch.float32: 소수점 계산(미분)을 위해 실수(float) 형식으로 만듭니다.
X_norm_tensor = torch.tensor(X_norm, dtype = torch.float32)
y_tensor = torch.tensor(y, dtype = torch.float32)

#   torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 형태,
#   즉 shape (n, 1) 이어야 합니다. 그래서 reshape(-1, 1)로 모양을 바꿉니다.
#       -1: 행 개수는 알아서 (여기서는 4)
#        1: 열 개수는 1 (입력 특성 1개)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print('학습용 입력 tensor X_norm_tensor:\n', X_norm_tensor)
print('학습용 정답 tensor y_tensor:\n', y_tensor)

#   shape을 꼭 확인합니다. 둘 다 (4, 1) 이어야 합니다.
print('X_norm_tensor shape: ', X_norm_tensor.shape)
print('y_tensor shape: ', y_tensor.shape)

학습용 입력 tensor X_norm_tensor:
 tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
학습용 정답 tensor y_tensor:
 tensor([[0.],
        [0.],
        [1.],
        [1.]])
X_norm_tensor shape:  torch.Size([4, 1])
y_tensor shape:  torch.Size([4, 1])


In [5]:
#   ========================================
#   3. PerceptronModel 정의 (torch.nn.Module 상속)
#   ========================================

#   torch.nn.Module을 상속받아 우리만의 모델 class를 만듭니다.
#   이전 파일에서 흩어져 있던 linear 생성과 H/z 계산을 이 안으로 모읍니다.
class PerceptronModel(torch.nn.Module):
    
    def __init__(self):
        #   super().__init__(): 부모 class인 torch.nn.Module의 초기화를 먼저 실행합니다.
        #                       이 줄이 있어야 PyTorch가 self.linear의 weight, bias를
        #                       '학습 대상 파라미터'로 제대로 등록하고 관리합니다.  (반드시 필요)
        super().__init__()
        
        #   self.linear       : 이 모델이 가지고 있는 선형 계산 부품입니다.
        #                       torch.nn.Linear(1, 1)은 입력 특성 1개를 받아 출력 1개(H 값)를 내보냅니다.
        #                       내부적으로 H(x) = a·X_norm + b를 계산하며,
        #                       기존 강의의 a는 self.linear.weight, b는 self.linear.bias 입니다.
        self.linear = torch.nn.Linear(1, 1)
        
    def forward(self, x):
        #   forward        : 입력 x가 들어왔을 때 실제 계산 순서를 정의하는 함수입니다.
        #                    이전 파일에서 학습 루프에 직접 적었던 두 줄이 바로 여기로 들어왔습니다.
        
        #   H(x) = a·X_norm + b (sigmoid 전의 선형 계산값 - 확률이 아닙니다.)
        H = self.linear(x)
        
        #   z = sigmoid(H)  (0~1 사이의 예측 확률)
        z = torch.sigmoid(H)
        
        #   이번 파일은 torch.nn.BCELoss()를 쓰므로, H가 아니라 z를 반환합니다.
        return z

In [ ]:
#   ========================================
#   3. model 생성   (linear를 단독으로 만들지 않습니다)
#   ========================================

#   torch.manual_seed(42)
#       - model 생성 시 내부의 self.linear.weight(=a), self.linear.bias(=b)가 랜덤 초기화되기 때문입니다.
torch.manual_seed(42)

#   model = PerceptronModel(): 우리가 정의한 class로 실제 모델 객체를 만듭니다.
#       - 이때 __init__이 실행되어 self.linear가 준비됩니다.
#       - 이전 파일처럼 linear를 단독으로 만들지 않고, model 안의 self.linear로 가지고 있습니다.
model = PerceptronModel()

#   model 안에 자동으로 만들어진 weight(=a)와 bias(=b)의 '학습 전' 초기값을 확인합니다.
#   model.linear.weight 는 기존 강의의 a 역할을 합니다.
#   model.linear.bias   는 기존 강의의 b 역할을 합니다.
print('model.linear.weight: ', model.linear.weight)
print('model.linear.bias: ', model.linear.bias)

#   model 전체 구조도 출력해 봅니다. 어떤 부품(self.linear)을 가지고 있는지 보여 줍니다.
print('model 구조: ', model)

model.linear.weight:  Parameter containing:
tensor([[0.7645]], requires_grad=True)
model.linear.bias:  Parameter containing:
tensor([0.8300], requires_grad=True)
model 구조:  PerceptronModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [7]:
#   ========================================
#   5. 학습 전 model 출력 확인 (구조 점검용)
#   ========================================

#   model(X_norm_tensor)를 호출하면 내부적으로 forward가 실행되어
#       H = self.linear(X_norm_tensor)
#       z = torch.sigmoid(H)
#   가 계산되고, 그 z가 반환됩니다.
#
#   아직 학습 전이라 weight, bias는 랜덤 초기값 상태입니다. (예측이 맞을 필요는 없p습니다.)
#   여기서는 '모델이 올바른 shape의 z를 내놓는가'만 확인합니다.
with torch.no_grad():
    z_test = model(X_norm_tensor)
    
#   z는 각 데이터의 예측 확률이므로 y_tensor와 같은 (4, 1) shape이어야 합니다.
print('z_test shape: ', z_test.shape)
print('z_test: \n', z_test)

z_test shape:  torch.Size([4, 1])
z_test: 
 tensor([[0.4512],
        [0.6197],
        [0.7635],
        [0.8648]])


In [8]:
#   ========================================
#   6. torch.nn.BCELoss() 생성  (이전 파일과 동일)
#   ========================================

#   torch.nn.BCELoss()
#       - Binary Cross Entropy Cost를 계산해 주는 PyTorch 부품입니다.
#       - 사용법: mean_cost = criterion(z, y_tensor) ← H가 아니라 z(sigmoid 통과값)를 넣습니다.
#       - 우리 model의 forward가 이미 z를 반환하므로, 그 z를 그대로 넣으면 됩니다.
criterion = torch.nn.BCELoss()

print('criterion 준비 완료: ', criterion)

criterion 준비 완료:  BCELoss()


In [9]:
#   ========================================
#   7. 학습 설정과 optimizer 생성
#   ========================================

#   learning_rate(학습률): 한 번에 weight, bias를 얼마나 크게 수정할지 정하는 값입니다.
learning_rate = 0.1

#   epochs(에폭): 전체 데이터를 몇 번 반복해서 학습할지 정하는 값입니다.
epochs = 1000

print('learning_rate: ', learning_rate)
print('epochs: ', epochs)

#   torch.optim.SGD(model.parameters(), lr= learning_rate)
#       - model.parameters(): optimizer가 업데이트할 학습 대상입니다.
#                             model 안의 self.linear.weight(=a)와 self.linear.bias(=b)를 가져옵니다.
#       - 이전 파일에서는 linear.parameters()를 넘겼지만,
#         이제 linear가 model 안에 있으므로 model.parameters()를 넘깁니다.
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

#   model.parameters()가 실제로 무엇을 가져오는지 눈으로 확인합니다.
#   이 출력 결과에 보이는 값들이 optimizer가 업데이트할 학습 대상입니다.
#   (첫 번째가 self.linear.weight(=a), 두 번째가 self.linear.bias(=b)입니다.)
print(list(model.parameters()))

learning_rate:  0.1
epochs:  1000
[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [10]:
#   ========================================
#   8. nn.Module 모델로 경사 하강법 학습    (이전 실습의 핵심 루프)
#   ========================================
#
#   한 번의 epoch에서 일어나는 단계 (반드시 이 순서):
#   1. optimizer.zero_grad()        : 이전 epoch의 grad를 0으로 초기화
#   2. z = model(X_norm_tensor)     : forward 실행(내부에서 H 계산 → sigmoid → z 반환)
#   3. mean_cost = criterion(z, y)  : torch.nn.BCELoss로 Cost 계산  (z를 넣음! H 아님)
#   4. mean_cost.backward()         : model 내부 weight.grad, bias.grad 자동 계산
#   5. optimizer.step()             : model 내부 weight, bias 업데이트
#   6. 학습 상태 출력

for epoch in range(epochs):
    #   ----- 1. 이전 epoch에서 계산된 grad 초기화 -----
    #   optimizer는 model.parameters()(= self.linear.weight, self.linear.bias)를 관리하므로,
    #   이 한 줄이 model 내부 두 파라미터의 grad를 한꺼번에 0으로 만듭니다.
    optimizer.zero_grad()
    
    #   ----- 2. model 호출 → 내부적으로 forward 실행 -----
    #   z = model(X_norm_tensor)
    #       H = self.linear(X_norm_tensor)  (H(x) = a·X_norm + b, 선형 계산값 - 확률 아님)
    #       z = torch.sigmoid(H)            (0~1 사이 예측 확률)
    #   를 실행한 결과입니다.   (외부에서 linear를 직접 호출하지 않습니다.)
    z = model(X_norm_tensor)
    
    #   ----- 3. torch.nn.BCELoss 로 Cost 계산 -----
    #   주의: criterion에는 H가 아니라, sigmoid를 통과한 z를 넣습니다.
    #        (우리 forward가 이미 z를 반환하므로 그 z를 그대로 사용합니다.)
    mean_cost = criterion(z, y_tensor)
    
    #   ----- 4. backward: model 내부 파라미터의 grad 자동 계산 -----
    #   Cost에서 출발해 model 내부 weight, bias까지 거꾸로 따라가며 미분값을 구해
    #       model.linear.weight.grad    (= 기존 grad_a)
    #       model.linear.bias.grad      (= 기존 grad_b)
    #   에 저장합니다.
    mean_cost.backward()
    
    #   ----- 5. optimizer가 model 내부 weight와 bias 업데이트 -----
    optimizer.step()
    
    #   ----- 6. 학습 상태 출력 -----
    #   입력 특성이 1개라 weight, bias에 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냅니다.
    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f'epoch = {epoch}, '
            f'Cost = {mean_cost.item():.6f}, '
            f'weight(a) = {model.linear.weight.item():.6f}, '
            f'bias(b) = {model.linear.bias.item():.6f}, '
        )
        
    #   (참고) 초반 3 epoch 에서만 grad 값을 확인해 봅니다.
    #   이전 파일에서는 linear.weight.grad, linear.bias.grad 를 봤지만,
    #   이제 linear가 model 안에 있으므로 model.linear.weight.grad, model.linear.bias.grad 를 봅니다.
    #       기존 grad_a → model.linear.weight.grad
    #       기존 grad_b → model.linear.bias.grad
    if epoch < 3:
        print(
            f'  (확인용) model.linear.weight.grad = {model.linear.weight.grad.item():.6f}'
            f'model.linear.bias.grad = {model.linear.bias.grad.item():.6f}'
        )

epoch = 0, Cost = 0.495464, weight(a) = 0.793780, bias(b) = 0.812529, 
  (확인용) model.linear.weight.grad = -0.292415model.linear.bias.grad = 0.174793
  (확인용) model.linear.weight.grad = -0.286153model.linear.bias.grad = 0.169918
  (확인용) model.linear.weight.grad = -0.280072model.linear.bias.grad = 0.165171
epoch = 100, Cost = 0.178670, weight(a) = 2.290082, bias(b) = 0.173212, 
epoch = 200, Cost = 0.125357, weight(a) = 3.002210, bias(b) = 0.061586, 
epoch = 300, Cost = 0.099283, weight(a) = 3.509002, bias(b) = 0.026837, 
epoch = 400, Cost = 0.082901, weight(a) = 3.912263, bias(b) = 0.013229, 
epoch = 500, Cost = 0.071398, weight(a) = 4.250606, bias(b) = 0.007116, 
epoch = 600, Cost = 0.062789, weight(a) = 4.543496, bias(b) = 0.004091, 
epoch = 700, Cost = 0.056068, weight(a) = 4.802371, bias(b) = 0.002480, 
epoch = 800, Cost = 0.050660, weight(a) = 5.034644, bias(b) = 0.001570, 
epoch = 900, Cost = 0.046207, weight(a) = 5.245449, bias(b) = 0.001031, 
epoch = 999, Cost = 0.042507, weight(a

In [11]:
#   ========================================
#   9. 학습 완료 후 최종 weight(a), bias(b) 확인
#   ========================================

#   학습된 weight, bias는 optimizer.step()에 의해 1000번 반복 업데이트된 값입니다.
#   입력 특성이 1개라 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냅니다.
print('학습된 weight(a): ', model.linear.weight.item())
print('학습된 bias(b): ', model.linear.bias.item())

# tensor 원본 형태도 함께 확인해 둡니다.    (shape과 requires_grad 표시를 볼 수 있습니다.)
print('model.linear.weight: ', model.linear.weight)
print('model.linear.bias: ', model.linear.bias)

#   학습 후 grad도 한 번 확인해 봅니다. (마지막 epoch의 grad가 남아 있습니다.)
#       기존 grad_a → model.linear.weight.grad
#       기존 grad_b → model.linear.bias.grad
print('model.linear.weight.grad: ', model.linear.weight.grad)
print('model.linear.bias.grad: ', model.linear.bias.grad)

학습된 weight(a):  5.436656951904297
학습된 bias(b):  0.0007013267604634166
model.linear.weight:  Parameter containing:
tensor([[5.4367]], requires_grad=True)
model.linear.bias:  Parameter containing:
tensor([0.0007], requires_grad=True)
model.linear.weight.grad:  tensor([[-0.0185]])
model.linear.bias.grad:  tensor([2.6396e-05])


In [ ]:
#   ========================================
#   10. 새로운 입력값 예측
#   ========================================

#   키가 185cm인 사람이 농구선수인지 예측합니다.
input_height = 185

#   새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 합니다.
#   학습 때 사용한 X_mean, X_std를 그대로 다시 사용합니다.  (새로 계산하면 안 됩니다.)
input_norm = (input_height - X_mean) / X_std

#   예측은 학습이 아니므로 weight, bias를 업데이트하지 않습니다.
#   따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산합니다.

with torch.no_grad():
    #   torch.nn.Linear(1, 1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 합니다.
    #       [[input_norm]] : 이중 대괄호로 감싸 (데이터 1개, 입력 특성 1개) - (1, 1) 형태로 만듭니다.
    input_norm_tensor = torch.tensor([[input_norm]], dtype = torch.float32)
    print('input_norm_tensor shape', input_norm_tensor.shape)
    
    #   z_new = model(input_norm_tensor) 는 내부적으로
    #       H_new = self.linear(input_norm_tensor)  (선형 계산값 - 확률 아님)
    #       z_new = torch.sigmoid(H_new)            (예측 확률 - 0 ~ 1 사이)
    #   를 실행한 결과입니다. H_new가 직접 보이지 않을 뿐, 계산은 그대로 일어납니다.
    z_new = model(input_norm_tensor)
    
    #   0.5 이상이면 1(농구선수), 미만이면 0(농구선수 아님)
    #   z_new는 shape(1, 1) tensor이므로 .item()으로 숫자 하나를 꺼냅니다.
    pred = 1 if z_new.item() >= 0.5 else 0
    
print(f'키가 {input_height}cm인 사람의 농구선수 확률(z): {z_new.item():.4f}')
if pred == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')

input_norm_tensor shape torch.Size([1, 1])
키가 185cm인 사람의 농구선수 확률(z): 0.9923
판별 결과: 농구선수입니다.


### 자습용 체크리스트1 - 개념 확인
- 이번 강의에서 수학 흐름은 바뀌었는가?
   기존 강의에서의 수학 흐름과 차이가 없습니다.
- torch.nn.Module은 무엇을 만들기 위한 기본 class인가?
   정규화 진행된 X_norm_tensor에 대한 선형회귀식 구축 후 최종적으로 시그모이드 결과 값을 산출하기 위한 class 입니다.
- __init__()에서는 무엇을 준비하는가?
   model() 실행 후, linear모델에서 활용할 weight(a)와 bias(b)를 준비합니다.
- self.linear는 기존 강의의 무엇에 해당하는가?
   객체 내 torch.nn.Linear() 역할을 담당합니다.
- forward()에서는 어떤 계산이 실행되는가?
   입력 데이터에 대한 회귀식 H 구축 후 z로 변환하는 과정이 실행됩니다.

### 자습용 체크리스트2 - model호출과 BCELoss 확인

#### model 호출 확인
- model(X_norm_tensor)를 호출하면 내부적으로 무엇이 실행되는가?
  X_norm_tensor에 대한 회귀식 H 생성 후 시그모이드 결과 z를 생성합니다.
- model(X_norm_tensor)의 출력은 H인가, z인가?
  시그모이드까지 반영된 z가 출력됩니다.
- model의 출력이 z인 이유는 무엇인가?
  class 내 forward에서 sigmoid까지 반영 후 z 값을 호출합니다.
- forward() 안에서 H는 어떻게 계산되는가?
  class 내 self.linear() 형태를 통해 계산됩니다.
- forward() 안에서 z는 어떻게 계산되는가?        
   생성된 회귀식 H에 대해 torch.sigmoid(H)를 통해 생성됩니다.

#### BCELoss와 optimizer 확인
- BECLoss에는 H를 넣어야 하는가, z를 넣어야 하는가?
   시그모이드 변형이 적용된 z를 넣어야 합니다.
- 기존 a, b는 이번 코드에서 각각 무엇인가?
   a는 model.linear.weight, b는 model.linear.bias 입니다.
- 기존 grad_a, grad_b는 이번 코드에서 각각 무엇인가?
   mean_cost.backward()를 통해 생성된 model.linear.weight.grad 및 model.linear.bias.grad입니다.
- linear.parameters()는 이번 코드에서 무엇으로 바뀌었는가?
   linear가 model 안에 있기 때문에, model.parameters()로 바뀌었습니다.
- optimizer.step()은 무엇을 업데이트하는가?
   model 내 weight와 bias를 업데이트합니다.

### 자습용 체크리스트3 - 예측과 정규화 확인

#### 예측 확인
- 새 데이터 예측에서는 왜 학습 때 사용한 X_mean, X_std를 그대로 사용해야 하는가?
  기존 모델에서 학습에서 활용한 데이터를 기준으로 사용해야 하기 때문입니다.
- 예측할 때 with torch.no_grad()를 사용하는 이유는 무엇인가?
  예측은 학습이 아니므로 weight 및 bias에 변화를 주지 않기 때문입니다.
- z_new가 0.5 이상이면 어떻게 판단하는가?
  농구선수로 판단합니다.
- z_new가 0.5 미만이면 어떻게 판단하는가?  
  농구선수가 아닌 걸로 판단합니다.
- 예측에서도 model 내부 흐름은 H → sigmoid → z 순서인가?
  예측에서도 model()을 통해 선형 회귀식 구축 및 시그모이드 변형이 진행됩니다.

  #### 전체 흐름 복습
- 학습 루프에서 optimizer.zero_grad()를 먼저 실행해야 하는 이유는 무엇인가?
  매 학습(epoch)마다 grad는 초기화되지 않고 누적되기 때문에 초기화를 시켜야 합니다.
- mean_cost.backward()는 무엇을 계산하는가?
  역전파를 의미하며 Cost에서 z, H를 거쳐 weight(a)와 bias(b)의 미분을 계산합니다.
- optimizer.step()은 무엇을 업데이트 하는가?
  weight(a)와 bias(b)을 업데이트합니다.